In [13]:
# Celda 1 – Librerías
import cv2
import numpy as np
import time
import customtkinter as ctk
from PIL import Image
import datetime

ctk.set_appearance_mode("dark")
print("Celda 1: Librerías cargadas correctamente.")

Celda 1: Librerías cargadas correctamente.


In [14]:
from pygrabber.dshow_graph import FilterGraph


def detectar_camaras_sistema():
    """Lee el registro de Windows sin encender el hardware."""
    try:
        graph = FilterGraph()
        return [(i, f'📷 {n}') for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar cámaras: {e}')
        return []


# ─── Rangos HSV estrictos (H: 0-179, S: 0-255, V: 0-255) ─────────────────────
RANGO_ROJO_1 = (np.array([  0, 150,  80]), np.array([  8, 255, 255]))
RANGO_ROJO_2 = (np.array([172, 150,  80]), np.array([180, 255, 255]))
RANGO_VERDE  = (np.array([ 45, 140,  80]), np.array([ 78, 255, 255]))
RANGO_AZUL   = (np.array([108, 150,  60]), np.array([128, 255, 255]))

# ─── Parámetros globales ──────────────────────────────────────────────────────
RATIO_COB_MIN = 0.25
FRAC_AREA_MIN = 0.005

KERNEL_OPEN  = np.ones((5, 5), np.uint8)
KERNEL_CLOSE = np.ones((7, 7), np.uint8)
KERNEL_ERODE = np.ones((3, 3), np.uint8)

# ─── Memoria de objetos ───────────────────────────────────────────────────────
# Umbral de similitud de histograma (0-1). Valores > UMBRAL_MISMO_OBJETO
# se consideran el mismo objeto ya registrado.
UMBRAL_MISMO_OBJETO = 0.82


def calcular_huella_hsv(frame_hsv, mascara, bbox):
    """
    Calcula la 'huella digital' de un objeto como histograma 2D (H x S)
    solo sobre los píxeles enmascarados dentro del bounding box.
    Retorna el histograma normalizado o None si no hay píxeles.
    """
    x, y, w, h = bbox
    roi_hsv  = frame_hsv[y:y+h, x:x+w]
    roi_mask = mascara[y:y+h, x:x+w]

    if cv2.countNonZero(roi_mask) < 50:
        return None

    # Histograma 2D: H (30 bins) x S (32 bins) → captura tono + saturación
    hist = cv2.calcHist(
        [roi_hsv], [0, 1], roi_mask,
        [30, 32], [0, 180, 0, 256]
    )
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist


def es_mismo_objeto(huella_nueva, huellas_guardadas, umbral=UMBRAL_MISMO_OBJETO):
    """
    Compara la huella nueva contra todas las huellas guardadas del mismo color.
    Retorna True si el objeto ya fue registrado anteriormente.
    """
    if huella_nueva is None or not huellas_guardadas:
        return False
    for huella_ref in huellas_guardadas:
        similitud = cv2.compareHist(huella_nueva, huella_ref, cv2.HISTCMP_CORREL)
        if similitud >= umbral:
            return True
    return False


class TrackerColor:
    """
    Estabiliza el bounding-box de un color detectado con EMA adaptativo,
    zona muerta y confirmación temporal.
    """

    def __init__(
        self,
        alpha_min      = 0.30,
        alpha_max      = 0.75,
        umbral_px      = 8,
        dist_max_px    = 80,
        frames_conf    = 2,
        frames_perdida = 4,
    ):
        self.alpha_min      = alpha_min
        self.alpha_max      = alpha_max
        self.umbral_px      = umbral_px
        self.dist_max_px    = dist_max_px
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.bbox_suave  = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, bbox_raw, frame_shape):
        h_f, w_f = frame_shape[:2]
        escala   = max(h_f, w_f) / 640.0
        umbral   = self.umbral_px   * escala
        dist_max = self.dist_max_px * escala

        if bbox_raw is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible    = False
                self.bbox_suave = None
            return self._int() if self.visible else None

        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)

        if self.conteo_det < self.frames_conf:
            return None

        if self.bbox_suave is None:
            self.bbox_suave = [float(v) for v in bbox_raw]
            self.visible    = True
            return self._int()

        self.visible = True

        cx_r = bbox_raw[0] + bbox_raw[2] / 2.0
        cy_r = bbox_raw[1] + bbox_raw[3] / 2.0
        cx_s = self.bbox_suave[0] + self.bbox_suave[2] / 2.0
        cy_s = self.bbox_suave[1] + self.bbox_suave[3] / 2.0
        dist = ((cx_r - cx_s) ** 2 + (cy_r - cy_s) ** 2) ** 0.5

        if dist < umbral:
            a = self.alpha_min * 0.4
            self.bbox_suave[2] = a * bbox_raw[2] + (1 - a) * self.bbox_suave[2]
            self.bbox_suave[3] = a * bbox_raw[3] + (1 - a) * self.bbox_suave[3]
        else:
            t     = min(1.0, dist / dist_max)
            alpha = self.alpha_min + t * (self.alpha_max - self.alpha_min)
            for i in range(4):
                self.bbox_suave[i] = (
                    alpha * bbox_raw[i] + (1 - alpha) * self.bbox_suave[i]
                )

        return self._int()

    def _int(self):
        if self.bbox_suave is None:
            return None
        return tuple(int(round(v)) for v in self.bbox_suave)


def calcular_orientacion(w, h):
    if h == 0 or w == 0:
        return 'Desconocido', 1.0
    ratio = h / w
    if ratio > 1.3:
        return 'Vertical', ratio
    elif ratio < 0.77:
        return 'Horizontal', ratio
    else:
        return 'Cuadrado', ratio


def estimar_profundidad_z(area_px, frame_shape):
    total_px = frame_shape[0] * frame_shape[1]
    frac     = area_px / total_px
    z = max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01)))
    return round(z, 2)


def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None):
    """
    Detecta Rojo / Verde / Azul, dibuja contornos precisos y bounding-boxes
    estabilizados. Calcula orientación X/Y/Z.
    callback_objeto(nombre, orientacion, x_norm, y_norm, z_est, huella_hist)
    """
    blurred       = cv2.GaussianBlur(frame, (5, 5), 0)
    hsv           = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    tiempo_actual = time.time()

    area_min = max(2000, int(frame.shape[0] * frame.shape[1] * FRAC_AREA_MIN))

    m_rojo = cv2.add(
        cv2.inRange(hsv, *RANGO_ROJO_1),
        cv2.inRange(hsv, *RANGO_ROJO_2),
    )
    m_verde = cv2.inRange(hsv, *RANGO_VERDE)
    m_azul  = cv2.inRange(hsv, *RANGO_AZUL)

    colores = [
        ('Rojo',  m_rojo,  (0,   0, 255)),
        ('Verde', m_verde, (0, 255,   0)),
        ('Azul',  m_azul,  (255, 0,   0)),
    ]

    for nombre, mascara, color_bgr in colores:
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN,  KERNEL_OPEN)
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, KERNEL_CLOSE)
        mascara_ref = cv2.erode(mascara, KERNEL_ERODE, iterations=1)
        mascara_ref = cv2.dilate(mascara_ref, KERNEL_ERODE, iterations=2)

        contornos, _ = cv2.findContours(
            mascara_ref, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS
        )

        bbox_raw        = None
        contorno_valido = None
        if contornos:
            c    = max(contornos, key=cv2.contourArea)
            area = cv2.contourArea(c)
            if area >= area_min:
                x, y, w, h = cv2.boundingRect(c)
                px_color   = cv2.countNonZero(mascara[y:y+h, x:x+w])
                if px_color / (w * h) >= RATIO_COB_MIN:
                    bbox_raw        = (x, y, w, h)
                    contorno_valido = c

        bbox_estable = trackers[nombre].actualizar(bbox_raw, frame.shape)

        if bbox_estable is None:
            temporizadores[nombre] = 0.0
            continue

        x, y, w, h = bbox_estable

        if temporizadores[nombre] == 0.0:
            temporizadores[nombre] = tiempo_actual

        label_y = max(18, y - 12)

        orientacion, ratio = calcular_orientacion(w, h)
        cx_px  = x + w / 2.0
        cy_px  = y + h / 2.0
        fh, fw = frame.shape[:2]
        x_norm = round((cx_px / fw - 0.5) * 2, 2)
        y_norm = round((0.5 - cy_px / fh) * 2, 2)
        z_est  = estimar_profundidad_z(w * h, frame.shape)

        if (tiempo_actual - temporizadores[nombre]) <= 3.0:
            paso = 15
            for i in range(x, x + w, paso):
                cv2.line(frame, (i, y), (i, y + h), color_bgr, 1)
            for j in range(y, y + h, paso):
                cv2.line(frame, (x, j), (x + w, j), color_bgr, 1)
            cv2.putText(frame, 'Calibrando...', (x, label_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
        else:
            if contorno_valido is not None:
                epsilon        = 0.008 * cv2.arcLength(contorno_valido, True)
                contorno_suave = cv2.approxPolyDP(contorno_valido, epsilon, True)
                cv2.drawContours(frame, [contorno_suave], -1, color_bgr, 2)
                cv2.rectangle(frame, (x, y), (x + w, y + h), color_bgr, 1)
            else:
                cv2.rectangle(frame, (x, y), (x + w, y + h), color_bgr, 2)

            flecha = '↕' if orientacion == 'Vertical' else ('↔' if orientacion == 'Horizontal' else '⊙')
            etiqueta_p = f'{nombre} {flecha}'
            etiqueta_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'

            (tw, th), _ = cv2.getTextSize(etiqueta_p, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (x, label_y - th - 4), (x + tw + 4, label_y + 4), (30, 30, 30), -1)
            cv2.putText(frame, etiqueta_p, (x + 2, label_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)

            coord_y = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(etiqueta_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (x, coord_y - ch - 2), (x + cw + 4, coord_y + 2), (30, 30, 30), -1)
            cv2.putText(frame, etiqueta_c, (x + 2, coord_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 220), 1)

            # ── Calcular huella y notificar a la UI ───────────────────────────
            if callback_objeto is not None:
                huella = calcular_huella_hsv(hsv, mascara, (x, y, w, h))
                callback_objeto(nombre, orientacion, x_norm, y_norm, z_est, huella)

    return frame


print('Celda 2: TrackerColor + detección compilados.')

Celda 2: TrackerColor + detección compilados.


In [15]:
# Celda 3 – UI
class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Sistema de Detección Profesional')
        self.geometry('1340x820')
        self.configure(fg_color='#1e2230')

        self.cap            = None
        self.escaneando     = False
        self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}

        self.trackers = {
            'Rojo':  TrackerColor(),
            'Verde': TrackerColor(),
            'Azul':  TrackerColor(),
        }

        # ── Memoria de objetos ya registrados ────────────────────────────────
        # Guarda la huella HSV de cada objeto único detectado por color.
        self.huellas_memoria = {'Rojo': [], 'Verde': [], 'Azul': []}

        # Historial visual (lista de dicts para la UI)
        self.historial_objetos = []

        # Cooldown por color para evitar llamadas repetidas en frames consecutivos
        # (la verificación real la hace la huella, esto solo evita llamadas excesivas)
        self.ultimo_intento   = {}
        self.COOLDOWN_INTENTO = 1.5   # segundos entre intentos de registro

        # Estado de si el objeto de cada color está «actualmente visible»
        # para saber cuándo reaparece tras haber salido de cuadro
        self.estaba_visible   = {'Rojo': False, 'Verde': False, 'Azul': False}

        self.construir_encabezado()
        self.construir_panel_config()
        self.construir_panel_video()

    # ── Construcción de la UI ─────────────────────────────────────────────────

    def construir_encabezado(self):
        self.lbl_titulo = ctk.CTkLabel(
            self, text='CONFIGURACIÓN DE CÁMARA',
            font=('Helvetica', 24, 'bold'), text_color='#e8eaf0'
        )
        self.lbl_titulo.pack(pady=(30, 15))

    def construir_panel_config(self):
        self.btn_detectar = ctk.CTkButton(
            self, text='🔍 DETECTAR CÁMARAS',
            fg_color='#3498db', hover_color='#2980b9',
            font=('Helvetica', 14, 'bold'), command=self.accion_buscar
        )
        self.btn_detectar.pack(pady=10)

        self.lbl_estado = ctk.CTkLabel(
            self, text='Haz clic para buscar dispositivos',
            text_color='#a0aab5', font=('Helvetica', 12)
        )
        self.lbl_estado.pack(pady=(0, 20))

        self.lbl_lista_tit = ctk.CTkLabel(
            self, text='Cámaras Disponibles:',
            font=('Helvetica', 14, 'bold'), text_color='#ffffff'
        )
        self.lbl_lista_tit.pack(anchor='w', padx=150)

        self.frame_lista = ctk.CTkScrollableFrame(
            self, fg_color='#2e3548', width=700, height=150,
            corner_radius=8, border_width=1, border_color='#4a5568'
        )
        self.frame_lista.pack(pady=5, padx=150)

        self.indice_seleccionado = ctk.StringVar(value='-1')

        self.lbl_pie_lista = ctk.CTkLabel(
            self, text='Ninguna cámara seleccionada',
            text_color='#a0aab5', font=('Helvetica', 12)
        )
        self.lbl_pie_lista.pack(anchor='w', padx=150)

        self.btn_iniciar = ctk.CTkButton(
            self, text='INICIAR DETECCIÓN',
            fg_color='#3b8ed0', hover_color='#296899',
            font=('Helvetica', 16, 'bold'), width=250, height=45,
            state='disabled', command=self.iniciar_camara
        )
        self.btn_iniciar.pack(pady=(40, 0))

    def construir_panel_video(self):
        self.frame_monitor = ctk.CTkFrame(self, fg_color='transparent')

        # ── Panel izquierdo: video ────────────────────────────────────────────
        self.frame_video_col = ctk.CTkFrame(self.frame_monitor, fg_color='transparent')
        self.frame_video_col.pack(side='left', fill='both', expand=True)

        self.lbl_video = ctk.CTkLabel(self.frame_video_col, text='')
        self.lbl_video.pack(pady=10, padx=10)

        self.frame_controles = ctk.CTkFrame(self.frame_video_col, fg_color='transparent')
        self.frame_controles.pack(pady=10)

        self.btn_escaneo = ctk.CTkButton(
            self.frame_controles, text='▶ Iniciar Escaneo',
            font=('Helvetica', 14, 'bold'),
            fg_color='#f39c12', hover_color='#d68910',
            command=self.toggle_escaneo
        )
        self.btn_escaneo.grid(row=0, column=0, padx=20)

        self.btn_detener = ctk.CTkButton(
            self.frame_controles, text='⏹ Detener Cámara',
            font=('Helvetica', 14, 'bold'),
            fg_color='#e74c3c', hover_color='#c0392b',
            command=self.apagar_hardware
        )
        self.btn_detener.grid(row=0, column=1, padx=20)

        # ── Panel derecho: lista de objetos ───────────────────────────────────
        self.frame_historial = ctk.CTkFrame(
            self.frame_monitor,
            fg_color='#252c3d',
            width=300,
            corner_radius=12,
            border_width=1,
            border_color='#3a4460'
        )
        self.frame_historial.pack(side='right', fill='y', padx=(0, 12), pady=10)
        self.frame_historial.pack_propagate(False)

        encab_frame = ctk.CTkFrame(self.frame_historial, fg_color='#1a2035', corner_radius=10)
        encab_frame.pack(fill='x', padx=8, pady=(8, 4))

        ctk.CTkLabel(
            encab_frame, text='📋 OBJETOS DETECTADOS',
            font=('Helvetica', 12, 'bold'), text_color='#c8d0e0'
        ).pack(pady=(8, 2))

        self.lbl_conteo = ctk.CTkLabel(
            encab_frame, text='Total: 0  |  Únicos en memoria: 0',
            font=('Helvetica', 9), text_color='#7a8aaa'
        )
        self.lbl_conteo.pack(pady=(0, 8))

        self.btn_limpiar = ctk.CTkButton(
            self.frame_historial,
            text='🗑 Limpiar lista',
            font=('Helvetica', 11),
            fg_color='#3a3f52', hover_color='#4a5268',
            height=28,
            command=self.limpiar_historial
        )
        self.btn_limpiar.pack(fill='x', padx=8, pady=(0, 4))

        self.lista_scroll = ctk.CTkScrollableFrame(
            self.frame_historial,
            fg_color='#1e2535',
            corner_radius=8
        )
        self.lista_scroll.pack(fill='both', expand=True, padx=8, pady=(0, 8))

    # ── Lógica de memoria y registro de objetos ───────────────────────────────

    COLORES_UI = {
        'Rojo':  ('#ff4d4d', '🔴'),
        'Verde': ('#4dff88', '🟢'),
        'Azul':  ('#4d9fff', '🔵'),
    }

    def on_objeto_detectado(self, nombre, orientacion, x_norm, y_norm, z_est, huella):
        """
        Callback por frame. Lógica:
        1. Cooldown de 1.5s entre intentos para evitar carga excesiva.
        2. Comparar huella con la memoria (huellas_memoria[nombre]).
        3. Si ya existe → solo actualizar indicador «ya conocido» en UI.
        4. Si es nuevo  → guardar huella, agregar a lista.
        """
        ahora  = time.time()
        ultimo = self.ultimo_intento.get(nombre, 0.0)
        if ahora - ultimo < self.COOLDOWN_INTENTO:
            return
        self.ultimo_intento[nombre] = ahora

        # ── Verificar memoria ─────────────────────────────────────────────────
        ya_conocido = es_mismo_objeto(huella, self.huellas_memoria[nombre])

        if ya_conocido:
            # Objeto ya registrado → NO agregar entrada nueva
            return

        # ── Objeto nuevo: guardar huella y registrar en UI ────────────────────
        if huella is not None:
            self.huellas_memoria[nombre].append(huella)

        num_color = sum(1 for o in self.historial_objetos if o['color'] == nombre) + 1
        total_memoria = sum(len(v) for v in self.huellas_memoria.values())

        entrada = {
            'color':       nombre,
            'num':         num_color,
            'orientacion': orientacion,
            'x':           x_norm,
            'y':           y_norm,
            'z':           z_est,
            'hora':        datetime.datetime.now().strftime('%H:%M:%S'),
        }
        self.historial_objetos.append(entrada)
        self.after(0, lambda e=entrada, tm=total_memoria: self._agregar_fila_historial(e, tm))

    def _agregar_fila_historial(self, entrada, total_memoria):
        """Añade una fila al panel lateral."""
        color_hex, emoji = self.COLORES_UI.get(entrada['color'], ('#aaaaaa', '⚪'))

        fila = ctk.CTkFrame(
            self.lista_scroll,
            fg_color='#2a3145',
            corner_radius=8,
            border_width=1,
            border_color='#3a4460'
        )
        fila.pack(fill='x', pady=3, padx=2)

        barra = ctk.CTkFrame(fila, fg_color=color_hex, width=5, corner_radius=4)
        barra.pack(side='left', fill='y', padx=(3, 6), pady=4)

        contenido = ctk.CTkFrame(fila, fg_color='transparent')
        contenido.pack(side='left', fill='both', expand=True, pady=4)

        ctk.CTkLabel(
            contenido,
            text=f'{emoji} {entrada["color"]} #{entrada["num"]}',
            font=('Helvetica', 12, 'bold'),
            text_color=color_hex,
            anchor='w'
        ).pack(anchor='w')

        icono_orient = '↕' if entrada['orientacion'] == 'Vertical' else ('↔' if entrada['orientacion'] == 'Horizontal' else '⊙')
        ctk.CTkLabel(
            contenido,
            text=f'{icono_orient} {entrada["orientacion"]}',
            font=('Helvetica', 10),
            text_color='#9aaac0',
            anchor='w'
        ).pack(anchor='w')

        ctk.CTkLabel(
            contenido,
            text=f'X:{entrada["x"]:+.2f}  Y:{entrada["y"]:+.2f}  Z:{entrada["z"]}m',
            font=('Courier', 9),
            text_color='#6a7a9a',
            anchor='w'
        ).pack(anchor='w')

        ctk.CTkLabel(
            contenido,
            text=f'🕐 {entrada["hora"]}',
            font=('Helvetica', 9),
            text_color='#4a5a72',
            anchor='w'
        ).pack(anchor='w')

        total_lista = len(self.historial_objetos)
        self.lbl_conteo.configure(
            text=f'Total: {total_lista}  |  En memoria: {total_memoria}'
        )
        self.lista_scroll._parent_canvas.yview_moveto(1.0)

    def limpiar_historial(self):
        """
        Limpia la lista visual Y la memoria de huellas, de modo que
        los objetos que estén frente a la cámara se vuelvan a detectar
        y registrar automáticamente.
        """
        # 1. Borrar widgets de la lista
        for widget in self.lista_scroll.winfo_children():
            widget.destroy()

        # 2. Resetear datos internos
        self.historial_objetos.clear()
        self.huellas_memoria   = {'Rojo': [], 'Verde': [], 'Azul': []}
        self.ultimo_intento    = {}   # forzar re-intento inmediato

        # 3. Resetear trackers para que los objetos reinicien ciclo de confirmación
        self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}
        for tracker in self.trackers.values():
            tracker.reiniciar()

        self.lbl_conteo.configure(text='Total: 0  |  En memoria: 0')

    # ── Lógica de la app ──────────────────────────────────────────────────────

    def accion_buscar(self):
        self.btn_detectar.configure(text='Buscando hardware...', state='disabled')
        self.update()
        for widget in self.frame_lista.winfo_children():
            widget.destroy()
        camaras = detectar_camaras_sistema()
        if camaras:
            self.lbl_estado.configure(
                text=f'Se encontraron {len(camaras)} cámara(s)', text_color='#2ecc71'
            )
            for indice_cv2, etiqueta in camaras:
                rb = ctk.CTkRadioButton(
                    self.frame_lista, text=etiqueta,
                    variable=self.indice_seleccionado,
                    value=str(indice_cv2), font=('Helvetica', 13)
                )
                rb.pack(anchor='w', pady=5, padx=10)
            self.indice_seleccionado.set(str(camaras[0][0]))
            self.lbl_pie_lista.configure(
                text='Selecciona la cámara que deseas usar y presiona Iniciar'
            )
            self.btn_iniciar.configure(state='normal')
        else:
            self.lbl_estado.configure(
                text='No se detectó hardware de video', text_color='#e74c3c'
            )
            self.lbl_pie_lista.configure(text='Asegúrate de que la cámara esté conectada')
            self.btn_iniciar.configure(state='disabled')
        self.btn_detectar.configure(text='🔍 DETECTAR CÁMARAS', state='normal')

    def iniciar_camara(self):
        indice = int(self.indice_seleccionado.get())
        if indice != -1:
            self.btn_detectar.pack_forget()
            self.lbl_estado.pack_forget()
            self.lbl_lista_tit.pack_forget()
            self.frame_lista.pack_forget()
            self.lbl_pie_lista.pack_forget()
            self.btn_iniciar.pack_forget()
            self.lbl_titulo.configure(text='MONITOR EN VIVO')
            self.frame_monitor.pack(expand=True, fill='both')
            self.cap = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
            self.escaneando = False
            self.btn_escaneo.configure(
                text='▶ Iniciar Escaneo', fg_color='#f39c12', hover_color='#d68910'
            )
            self.loop_video()

    def toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando:
            self.btn_escaneo.configure(
                text='⏸ Pausar Escaneo', fg_color='#d35400', hover_color='#a04000'
            )
            self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}
            for tracker in self.trackers.values():
                tracker.reiniciar()
        else:
            self.btn_escaneo.configure(
                text='▶ Reanudar Escaneo', fg_color='#f39c12', hover_color='#d68910'
            )

    def loop_video(self):
        if self.cap and self.cap.isOpened():
            ret, frame = self.cap.read()
            if ret:
                frame = cv2.flip(frame, 1)
                if self.escaneando:
                    frame = procesar_frame_vision(
                        frame,
                        self.temporizadores,
                        self.trackers,
                        callback_objeto=self.on_objeto_detectado
                    )
                img = ctk.CTkImage(
                    light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)),
                    size=(900, 620)
                )
                self.lbl_video.configure(image=img)
            self.after(15, self.loop_video)

    def apagar_hardware(self):
        if self.cap:
            self.cap.release()
            self.cap = None
        self.escaneando = False
        self.frame_monitor.pack_forget()
        self.lbl_titulo.configure(text='CONFIGURACIÓN DE CÁMARA')
        self.btn_detectar.pack(pady=10)
        self.lbl_estado.pack(pady=(0, 20))
        self.lbl_lista_tit.pack(anchor='w', padx=150)
        self.frame_lista.pack(pady=5, padx=150)
        self.lbl_pie_lista.pack(anchor='w', padx=150)
        self.btn_iniciar.pack(pady=(40, 0))

    def destroy(self):
        self.apagar_hardware()
        super().destroy()


print('Celda 3: UI con trackers + memoria de objetos lista.')

Celda 3: UI con trackers + memoria de objetos lista.


In [16]:
# Celda 4 – Punto de entrada
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()